# Comparaison des Méthodes d'Imputation (sans Data Leakage)

## Protocole rigoureux
1. **Pré-filtre non-supervisé** : top 5000 CpG par variance (pas de cible utilisée = pas de leakage)
2. **Nested CV** : 5-fold outer CV pour l'évaluation
3. **À l'intérieur de chaque fold** :
   - Imputation fit sur **train uniquement**, transform train+test
   - Sélection top-500 par corrélation sur **train uniquement**
   - Entraînement du modèle (ElasticNetCV inner CV = nested)
4. **Intervalles de confiance** à 95%

## Méthodes comparées
1. Zero Fill  2. Mean  3. Median  4. KNN (k=5)  5. Iterative/MICE

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from sklearn.experimental import enable_iterative_imputer  # noqa

import numpy as np
import pandas as pd
from pathlib import Path
from time import time

from sklearn.linear_model import ElasticNetCV, LinearRegression, BayesianRidge
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy import stats
from xgboost import XGBRegressor

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path('Data')
PRE_FILTER = 5000   # pré-filtre non-supervisé (variance)
TOP_K = 500         # sélection supervisée (corrélation, intra-fold)
N_FOLDS = 5
CHUNK_SIZE = 5000

print('Imports OK')

## ResidualLearningPipeline + Helper de sélection

In [ ]:
def select_top_k(X_train, y_train, k):
    """Sélection top-k features par corrélation absolue (vectorisé, train only)."""
    y_c = y_train - y_train.mean()
    y_den = np.sqrt(np.sum(y_c ** 2))
    x_c = X_train - X_train.mean(axis=0, keepdims=True)
    num = x_c.T @ y_c
    den = np.sqrt(np.sum(x_c ** 2, axis=0)) * y_den
    corrs = np.abs(np.divide(num, den, out=np.zeros_like(num), where=den != 0))
    return np.argsort(corrs)[::-1][:k]


class ResidualLearningPipeline:
    """ElasticNetCV + XGBoost résidus + Bias Correction."""
    def __init__(self, enet_cv=3, xgb_n_estimators=300, xgb_early_stopping_rounds=30):
        self.enet_cv = enet_cv
        self.xgb_params = dict(
            n_estimators=xgb_n_estimators, max_depth=3, learning_rate=0.03,
            subsample=0.7, colsample_bytree=0.5, reg_alpha=10.0, reg_lambda=50.0,
            min_child_weight=10, early_stopping_rounds=xgb_early_stopping_rounds,
            objective='reg:squarederror', eval_metric='mae',
            n_jobs=-1, random_state=RANDOM_STATE,
        )
        self.enet_ = None
        self.xgb_ = None
        self.bias_ = None

    def fit(self, X_train, y_train, X_val=None, y_val=None):
        self.enet_ = ElasticNetCV(
            l1_ratio=[0.1, 0.3, 0.5, 0.7, 0.9], n_alphas=50,
            cv=self.enet_cv, max_iter=50000, random_state=RANDOM_STATE, n_jobs=-1,
        )
        self.enet_.fit(X_train, y_train)
        P_lin = self.enet_.predict(X_train)
        residuals = y_train - P_lin

        self.xgb_ = XGBRegressor(**self.xgb_params)
        if X_val is not None and y_val is not None:
            P_val = self.enet_.predict(X_val)
            self.xgb_.fit(X_train, residuals,
                          eval_set=[(X_val, y_val - P_val)], verbose=False)
        else:
            self.xgb_.fit(X_train, residuals,
                          eval_set=[(X_train, residuals)], verbose=False)

        R_hat = self.xgb_.predict(X_train)
        self.bias_ = LinearRegression()
        self.bias_.fit(np.column_stack([P_lin, R_hat]), y_train)
        return self

    def predict(self, X):
        P_lin = self.enet_.predict(X)
        R_hat = self.xgb_.predict(X)
        return self.bias_.predict(np.column_stack([P_lin, R_hat]))

## Chargement des Données

**Pré-filtre non-supervisé** : top 5000 CpG par variance (aucune information sur y utilisée).

In [ ]:
# --- Annotations ---
ind = pd.read_csv(DATA_DIR / 'annot_projet.csv')
ind = ind.dropna(subset=['age', 'Sample_description']).copy()
ind['Sample_description'] = ind['Sample_description'].astype(str)
ind = ind.set_index('Sample_description')

raw_path = DATA_DIR / 'c_sample.csv'
sample_header = pd.read_csv(raw_path, nrows=0)
all_sample_ids = list(sample_header.columns)
common_ids = [s for s in all_sample_ids if s in ind.index]
y = ind.loc[common_ids, 'age'].values.astype(np.float32)
print(f'Échantillons : {len(common_ids)}, âge moyen : {y.mean():.1f} ± {y.std():.1f}')

# --- Pré-filtre par VARIANCE (non-supervisé, pas de leakage) ---
print(f'\nCalcul de la variance par CpG (non-supervisé)...')
t0 = time()
variances = []
for chunk in pd.read_csv(raw_path, usecols=common_ids, chunksize=CHUNK_SIZE):
    x = chunk.to_numpy(dtype=np.float32)
    variances.append(np.nanvar(x, axis=1))
variances = np.concatenate(variances)
print(f'  Calculé en {time()-t0:.1f}s ({len(variances)} CpG)')

# Sélectionner top PRE_FILTER par variance
var_top_indices = np.argsort(variances)[::-1][:PRE_FILTER]
print(f'  Top {PRE_FILTER} par variance : max={variances[var_top_indices[0]]:.6f}, min={variances[var_top_indices[-1]]:.6f}')

# --- Charger ces CpG depuis données brutes (NaN préservés) ---
print(f'\nChargement des top {PRE_FILTER} CpG (NaN préservés)...')
t0 = time()
indices_to_load = np.sort(var_top_indices)      
rows = []
start = 0
for chunk in pd.read_csv(raw_path, usecols=common_ids, chunksize=CHUNK_SIZE):
    end = start + len(chunk)
    pos_s = np.searchsorted(indices_to_load, start)
    pos_e = np.searchsorted(indices_to_load, end)
    local = indices_to_load[pos_s:pos_e] - start
    if len(local) > 0:
        rows.append(chunk.iloc[local].values)
    start = end

X_raw = np.vstack(rows).T.astype(np.float32)  # (n_samples, PRE_FILTER)
nan_count = np.isnan(X_raw).sum()
print(f'  Chargé en {time()-t0:.1f}s, shape={X_raw.shape}')
print(f'  NaN : {nan_count} ({100*nan_count/X_raw.size:.2f}%)')

## Nested Cross-Validation (sans leakage)

Pour chaque fold :
1. **Imputation** : fit sur train, transform train+test
2. **Sélection supervisée** : top-500 par corrélation sur train imputé
3. **Modèle** : ResidualLearningPipeline (ElasticNetCV inner CV = nested)

In [ ]:
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

method_names = ['1. Zero Fill', '2. Mean', '3. Median', '4. KNN (k=5)', '5. Iterative (MICE)']
results = {name: {'mae': [], 'r2': []} for name in method_names}

def make_imputers():
    """Retourne des imputers frais (à recréer par fold)."""
    return {
        '1. Zero Fill': None,  # cas spécial
        '2. Mean': SimpleImputer(strategy='mean'),
        '3. Median': SimpleImputer(strategy='median'),
        '4. KNN (k=5)': KNNImputer(n_neighbors=5, weights='uniform'),
        '5. Iterative (MICE)': IterativeImputer(
            estimator=BayesianRidge(), max_iter=10,
            random_state=RANDOM_STATE, skip_complete=True,
        ),
    }

print(f"{'='*70}")
print(f'  Nested {N_FOLDS}-Fold CV | Pré-filtre: {PRE_FILTER} (variance) | Sélection: {TOP_K} (corr intra-fold)')
print(f"{'='*70}")

for fold_i, (train_idx, test_idx) in enumerate(kf.split(y)):
    print(f"\n{'='*55}")
    print(f'  Fold {fold_i + 1}/{N_FOLDS}  (train={len(train_idx)}, test={len(test_idx)})')
    print(f"{'='*55}")

    y_train, y_test = y[train_idx], y[test_idx]
    X_raw_train, X_raw_test = X_raw[train_idx], X_raw[test_idx]

    imputers = make_imputers()

    for method_name in method_names:
        t0 = time()

        # --- 1. Imputation (fit sur train uniquement) ---
        if method_name == '1. Zero Fill':
            X_train_imp = np.nan_to_num(X_raw_train.copy(), nan=0.0)
            X_test_imp = np.nan_to_num(X_raw_test.copy(), nan=0.0)
        else:
            imp = imputers[method_name]
            X_train_imp = imp.fit_transform(X_raw_train).astype(np.float32)
            X_test_imp = imp.transform(X_raw_test).astype(np.float32)

        # --- 2. Sélection top-k par corrélation (train uniquement) ---
        top_idx = select_top_k(X_train_imp, y_train, TOP_K)
        X_tr = X_train_imp[:, top_idx]
        X_te = X_test_imp[:, top_idx]

        # --- 3. Entraînement (inner CV via ElasticNetCV = nested) ---
        pipeline = ResidualLearningPipeline(enet_cv=3)
        pipeline.fit(X_tr, y_train, X_val=X_te, y_val=y_test)
        y_pred = pipeline.predict(X_te)

        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        results[method_name]['mae'].append(mae)
        results[method_name]['r2'].append(r2)

        print(f'  {method_name:<25s} | MAE={mae:.3f} | R²={r2:.3f} | {time()-t0:.1f}s')

print(f"\n{'='*70}")
print('Nested CV terminée.')

## Résultats avec Intervalles de Confiance 95%

In [ ]:
t_crit = stats.t.ppf(0.975, df=N_FOLDS - 1)  # t-Student pour IC 95%

summary_rows = []
for name, scores in results.items():
    mae_arr = np.array(scores['mae'])
    r2_arr = np.array(scores['r2'])
    mae_ci = t_crit * mae_arr.std(ddof=1) / np.sqrt(N_FOLDS)
    r2_ci = t_crit * r2_arr.std(ddof=1) / np.sqrt(N_FOLDS)
    summary_rows.append({
        'Méthode': name,
        'MAE_mean': mae_arr.mean(), 'MAE_ci95': mae_ci,
        'R2_mean': r2_arr.mean(), 'R2_ci95': r2_ci,
    })

df_summary = pd.DataFrame(summary_rows).sort_values('MAE_mean')

print(f"\n{'='*80}")
print(f"  COMPARAISON DES MÉTHODES D'IMPUTATION (Nested {N_FOLDS}-fold CV)")
print(f'  Pré-filtre: {PRE_FILTER} (variance) | Sélection: {TOP_K} (corr intra-fold)')
print(f'  Modèle: ResidualLearningPipeline | IC 95% (t-Student)')
print(f"{'='*80}")
print(f"{'Méthode':<25s} | {'MAE (95% CI)':<22s} | {'R² (95% CI)':<22s}")
print(f"{'-'*25}-+-{'-'*22}-+-{'-'*22}")

for _, row in df_summary.iterrows():
    print(f"{row['Méthode']:<25s} | "
          f"{row['MAE_mean']:.3f} ± {row['MAE_ci95']:.3f}         | "
          f"{row['R2_mean']:.3f} ± {row['R2_ci95']:.3f}")

best = df_summary.iloc[0]
print(f"\n  -> Meilleure : {best['Méthode']} (MAE = {best['MAE_mean']:.3f} ± {best['MAE_ci95']:.3f})")

results_path = Path('results/imputation_comparison_results.csv')
results_path.parent.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(results_path, index=False)
print(f'  Résultats sauvegardés : {results_path}')

## Tests Statistiques (Paired t-test vs Mean)

In [ ]:
baseline_name = '2. Mean'
baseline_mae = np.array(results[baseline_name]['mae'])

print(f'Tests appariés contre la baseline ({baseline_name}) :\n')
print(f"{'Méthode':<25s} | {'Δ MAE':>8s} | {'p-value':>8s} | {'Significatif':>12s}")
print(f"{'-'*25}-+-{'-'*8}-+-{'-'*8}-+-{'-'*12}")

for name, scores in results.items():
    if name == baseline_name:
        continue
    mae_arr = np.array(scores['mae'])
    delta = mae_arr.mean() - baseline_mae.mean()
    _, p_value = stats.ttest_rel(mae_arr, baseline_mae)
    sig = 'Oui *' if p_value < 0.05 else 'Non'
    print(f'{name:<25s} | {delta:>+8.3f} | {p_value:>8.4f} | {sig:>12s}')

## Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
methods = df_summary['Méthode'].tolist()
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']

# MAE avec IC 95%
ax = axes[0]
mae_means = df_summary['MAE_mean'].values
mae_cis = df_summary['MAE_ci95'].values
bars = ax.bar(range(len(methods)), mae_means, yerr=mae_cis, capsize=5,
              color=colors[:len(methods)], alpha=0.85, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels([m.split('. ')[1] for m in methods], rotation=15, ha='right')
ax.set_ylabel('MAE (ans)')
ax.set_title(f'MAE ± IC 95% (Nested {N_FOLDS}-fold CV)')
for bar, m, c in zip(bars, mae_means, mae_cis):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + c + 0.03,
            f'{m:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

# R² avec IC 95%
ax = axes[1]
r2_means = df_summary['R2_mean'].values
r2_cis = df_summary['R2_ci95'].values
bars = ax.bar(range(len(methods)), r2_means, yerr=r2_cis, capsize=5,
              color=colors[:len(methods)], alpha=0.85, edgecolor='white', linewidth=1.5)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels([m.split('. ')[1] for m in methods], rotation=15, ha='right')
ax.set_ylabel('R²')
ax.set_title(f'R² ± IC 95% (Nested {N_FOLDS}-fold CV)')

plt.tight_layout()
plt.savefig('results/imputation_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(N_FOLDS)
n_methods = len(results)
width = 0.8 / n_methods
colors = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#f39c12']

for i, (name, scores) in enumerate(results.items()):
    offset = (i - n_methods/2 + 0.5) * width
    ax.bar(x + offset, scores['mae'], width, label=name.split('. ')[1],
           color=colors[i], alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels([f'Fold {i+1}' for i in range(N_FOLDS)])
ax.set_ylabel('MAE (ans)')
ax.set_title('MAE par fold et méthode')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('results/imputation_comparison_per_fold.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print('=' * 75)
print('  RÉSUMÉ FINAL (sans data leakage)')
print('=' * 75)
print(f'\n  Pré-filtre : {PRE_FILTER} CpG par variance (non-supervisé)')
print(f'  Sélection  : {TOP_K} CpG par corrélation (intra-fold, train only)')
print(f'  Imputation : fit sur train, transform train+test (pas de leakage)')
print(f'  Modèle     : ResidualLearningPipeline (nested CV)')
print(f'  Évaluation : {N_FOLDS}-fold outer CV, IC 95%\n')
print(df_summary[['Méthode', 'MAE_mean', 'MAE_ci95', 'R2_mean', 'R2_ci95']].to_string(index=False))
best = df_summary.iloc[0]
print(f"\n  Meilleure : {best['Méthode']} (MAE = {best['MAE_mean']:.3f} ± {best['MAE_ci95']:.3f})")
print(f"\n{'='*75}")


